# Props Data Organizer

## Definitions of Formulas Used in This Notebook

### Estimated Value (EV)
EV is the average amount you can expect to win or lose per bet if you placed the same bet many times. It helps identify profitable betting opportunities by comparing the expected return to the risk involved.

**Formula:**
$$
\text{EV} = (\text{Probability of Winning} \times \text{Profit if Win}) - (\text{Probability of Losing} \times \text{Loss if Lose})
$$

### Kelly Criterion
The Kelly Criterion is a formula used to determine the optimal size of a series of bets. It aims to maximize the logarithm of wealth, balancing the trade-off between risk and reward. The formula considers both the probability of winning and the odds offered, guiding you on how much of your bankroll to wager on each bet.

**Formula:**
$$
\text{Kelly Fraction} = \frac{(\text{Probability of Winning} \times (\text{Odds} + 1)) - 1}{\text{Odds}}
$$

### Variance
Variance in sports betting represents the spread or dispersion of actual outcomes around the expected value. It's a crucial metric for understanding the risk and volatility associated with betting predictions. Higher variance indicates more volatile and unpredictable outcomes, while lower variance suggests more consistent results.

**Formula:**
$$
\text{Variance} = \frac{\sum_{i=1}^{n} (x_i - \mu)^2}{n}
$$

Where:
- $x_i$ represents each individual outcome
- $\mu$ is the mean or expected value
- $n$ is the total number of observations

In the context of prop betting:
- High variance props (e.g., 3-pointers made) tend to be more risky but potentially more profitable
- Low variance props (e.g., minutes played) typically offer more consistent but lower returns


In [1]:
import pandas as pd 
import numpy as np
import time
import requests
import os
import sys
from datetime import datetime
import joblib

# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)
    
from PROPS_EV.calculateEVS import *
from MODELS.model import *

today = datetime.now()
formatted_date = today.strftime("%m_%d_%y")
pd.set_option('display.max_columns', None)

/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Grabs players odds for the day (US all boookmakers, DFS is prizepicks and underdogs)

In [2]:
# from NBAPropFinder.NBAPropFinder import NBAPropFinder

# nba_props = NBAPropFinder(region='us_dfs')
# prizePicks = nba_props.dataframe
# prizePicks.head(10)

### Single Bets from bookmakers that dont include prizePicks or UnderDogs

In [7]:
features = [
    # Player context
    'PLAYER_ID', 'TEAM_ID', 'OPP_TEAM_ID', 
    'STARTING', 'HOME_GAME', 
    'PLAYER_DAYS_REST', 'IS_BACK_TO_BACK', 
    
    # Player season averages
    'PTS_AVG_TO_DATE', 'MIN_AVG_TO_DATE', 'FGA_AVG_TO_DATE', 'FTA_AVG_TO_DATE', 'FG3A_AVG_TO_DATE', 
    'FG_PCT_AVG_TO_DATE', 'FG3_PCT_AVG_TO_DATE', 'FT_PCT_AVG_TO_DATE', 'USG_PCT_AVG_TO_DATE', 'TS_PCT_AVG_TO_DATE', 
    'EFG_PCT_AVG_TO_DATE', 'POSS_AVG_TO_DATE', 'TCHS_AVG_TO_DATE', 'AST_AVG_TO_DATE', 'REB_AVG_TO_DATE', 'TOV_AVG_TO_DATE',
    
    # LAG
    'PTS_LAG_1', 'PTS_LAG_2', 'MIN_LAG_1', 'MIN_LAG_2', 'FGA_LAG_1', 'FGA_LAG_2', 'FTA_LAG_1', 'FTA_LAG_2', 'FG3A_LAG_1', 
    'FG3A_LAG_2', 'FG_PCT_LAG_1', 'FG_PCT_LAG_2', 'FG3_PCT_LAG_1', 'FG3_PCT_LAG_2', 'FT_PCT_LAG_1', 'FT_PCT_LAG_2', 
    'USG_PCT_LAG_1', 'USG_PCT_LAG_2', 'TS_PCT_LAG_1', 'TS_PCT_LAG_2', 'EFG_PCT_LAG_1', 'EFG_PCT_LAG_2', 'POSS_LAG_1', 'POSS_LAG_2', 
    'TCHS_LAG_1', 'TCHS_LAG_2', 'AST_LAG_1', 'AST_LAG_2', 'REB_LAG_1', 'REB_LAG_2', 'TOV_LAG_1', 'TOV_LAG_2',
    
    # 3 game rolling averages
    'PTS_ROLLING_AVG_3', 'MIN_ROLLING_AVG_3', 'FGA_ROLLING_AVG_3', 'FTA_ROLLING_AVG_3', 'FG3A_ROLLING_AVG_3', 'FG_PCT_ROLLING_AVG_3', 
    'FG3_PCT_ROLLING_AVG_3', 'FT_PCT_ROLLING_AVG_3', 'USG_PCT_ROLLING_AVG_3', 'TS_PCT_ROLLING_AVG_3', 'EFG_PCT_ROLLING_AVG_3', 
    'POSS_ROLLING_AVG_3', 'TCHS_ROLLING_AVG_3', 'AST_ROLLING_AVG_3', 'REB_ROLLING_AVG_3', 'TOV_ROLLING_AVG_3',

    # Short-term form (5-game rolling averages)
    'PTS_ROLLING_AVG_5', 'MIN_ROLLING_AVG_5', 'FGA_ROLLING_AVG_5', 'FTA_ROLLING_AVG_5', 'FG3A_ROLLING_AVG_5', 'FG_PCT_ROLLING_AVG_5', 
    'FG3_PCT_ROLLING_AVG_5', 'FT_PCT_ROLLING_AVG_5', 'USG_PCT_ROLLING_AVG_5', 'TS_PCT_ROLLING_AVG_5', 'EFG_PCT_ROLLING_AVG_5', 
    'POSS_ROLLING_AVG_5', 'TCHS_ROLLING_AVG_5', 'AST_ROLLING_AVG_5', 'REB_ROLLING_AVG_5', 'TOV_ROLLING_AVG_5',
    
    # 7 game rolling averages
    'PTS_ROLLING_AVG_7', 'MIN_ROLLING_AVG_7', 'FGA_ROLLING_AVG_7', 'FTA_ROLLING_AVG_7', 'FG3A_ROLLING_AVG_7', 'FG_PCT_ROLLING_AVG_7', 
    'FG3_PCT_ROLLING_AVG_7', 'FT_PCT_ROLLING_AVG_7', 'USG_PCT_ROLLING_AVG_7', 'TS_PCT_ROLLING_AVG_7', 'EFG_PCT_ROLLING_AVG_7', 
    'POSS_ROLLING_AVG_7', 'TCHS_ROLLING_AVG_7', 'AST_ROLLING_AVG_7', 'REB_ROLLING_AVG_7', 'TOV_ROLLING_AVG_7',

    # Medium-term form (15-game rolling averages)
    'PTS_ROLLING_AVG_15', 'MIN_ROLLING_AVG_15', 'FGA_ROLLING_AVG_15', 'FTA_ROLLING_AVG_15', 'FG3A_ROLLING_AVG_15', 'FG_PCT_ROLLING_AVG_15', 
    'FG3_PCT_ROLLING_AVG_15', 'FT_PCT_ROLLING_AVG_15', 'USG_PCT_ROLLING_AVG_15', 'TS_PCT_ROLLING_AVG_15', 'EFG_PCT_ROLLING_AVG_15', 
    'POSS_ROLLING_AVG_15', 'TCHS_ROLLING_AVG_15', 'AST_ROLLING_AVG_15', 'REB_ROLLING_AVG_15', 'TOV_ROLLING_AVG_15',
    
    # Long-term form (40-game rolling averages)
    'PTS_ROLLING_AVG_40', 'MIN_ROLLING_AVG_40', 'FGA_ROLLING_AVG_40', 'FTA_ROLLING_AVG_40', 'FG3A_ROLLING_AVG_40', 'FG_PCT_ROLLING_AVG_40', 
    'FG3_PCT_ROLLING_AVG_40', 'FT_PCT_ROLLING_AVG_40', 'USG_PCT_ROLLING_AVG_40', 'TS_PCT_ROLLING_AVG_40', 'EFG_PCT_ROLLING_AVG_40', 
    'POSS_ROLLING_AVG_40', 'TCHS_ROLLING_AVG_40', 'AST_ROLLING_AVG_40', 'REB_ROLLING_AVG_40', 'TOV_ROLLING_AVG_40',

    # Opponent
    'OPP_DEF_RATING_AVG_TO_DATE', 'OPP_PACE_AVG_TO_DATE', 'OPP_PTS_AVG_TO_DATE', 'OPP_FGA_AVG_TO_DATE', 
    'OPP_REB_AVG_TO_DATE', 'OPP_AST_AVG_TO_DATE', 'OPP_TOV_AVG_TO_DATE', 'OPP_BLK_AVG_TO_DATE', 'OPP_STL_AVG_TO_DATE',
    
    #starters 
    'TEAM_OFF_RATING_AVG_TO_DATE','TEAM_DEF_RATING_AVG_TO_DATE','TEAM_PACE_AVG_TO_DATE', 'TEAM_FGA_AVG_TO_DATE',
    'TEAM_PTS_AVG_TO_DATE', 'TEAM_REB_AVG_TO_DATE', 'TEAM_AST_AVG_TO_DATE', 'TEAM_TOV_AVG_TO_DATE',
    
    # Team odds
    'team_spread', 'total', 'team_is_favored','TEAM_IMPLIED_PTS_FAV','TEAM_IMPLIED_PTS_UND','BLOWOUT_RISK'
]

model = joblib.load('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/Models/PTS_cat_model.pkl')
# model = joblib.load(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS\Models\PTS_cat_model.pkl")
data = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values('GAME_DATE', ascending=False)
bookmakers = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')

date = '2024-11-12'
espnDate = '20241112'
odds = bookmakers[
    (bookmakers['CATEGORY'] == 'points') &
    (bookmakers['GAME_DATE'] == date) &
    (bookmakers['ODDS'] < 200) &
    (bookmakers['ODDS'] > -200)
]

oddsPP = prizePicks[
    (prizePicks['CATEGORY'] == 'player_points') &
    (prizePicks['GAME_DATE'] == date) &
    (prizePicks['BOOKMAKER'] == 'underdog')
]
oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)
filterData = data[data['GAME_DATE'] <= date].sort_values('GAME_DATE', ascending=True)
games = get_espn_games(date_str=espnDate)
oddsPP.head()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_56616/3061805351.py:59: DtypeWarning: Columns (10,11,13) have mixed types. Specify dtype option on import or set low_memory=False.
  prizePicks = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/odds25.csv')
/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_56616/3061805351.py:75: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  oddsPP.rename(columns={'OVER/UNDER': 'SIDE'}, inplace=True)


,Unnamed: 0.1,Unnamed: 0,NAME,CATEGORY,BOOKMAKER,SIDE,LINE,PRICE,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds,OVER/ODDS
12626,12626,12626,Tobias Harris,player_points,underdog,Over,13.5,-137,Detroit Pistons,Miami Heat,e80e0c514aa69872360f58b079796a8b,2024-11-13T00:10:00Z,2024-11-12,NaN,NaN,NaN,over
12627,12627,12627,Cade Cunningham,player_points,underdog,Over,22.0,-137,Detroit Pistons,Miami Heat,e80e0c514aa69872360f58b079796a8b,2024-11-13T00:10:00Z,2024-11-12,NaN,NaN,NaN,over
12628,12628,12628,Cade Cunningham,player_points,underdog,Under,22.0,-137,Detroit Pistons,Miami Heat,e80e0c514aa69872360f58b079796a8b,2024-11-13T00:10:00Z,2024-11-12,NaN,NaN,NaN,under
12629,12629,12629,Bam Adebayo,player_points,underdog,Under,18.5,-137,Detroit Pistons,Miami Heat,e80e0c514aa69872360f58b079796a8b,2024-11-13T00:10:00Z,2024-11-12,NaN,NaN,NaN,under
12630,12630,12630,Tyler Herro,player_points,underdog,Over,22.5,-137,Detroit Pistons,Miami Heat,e80e0c514aa69872360f58b079796a8b,2024-11-13T00:10:00Z,2024-11-12,NaN,NaN,NaN,over


## Best EVs for Single Bets from draftkings, fanduel, prizepicks, and underdog

In [8]:
final_results = single_bet(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)

print("\nTop 10 highest EV bets across point props:")
final_results.head(10)

Processing single bets...

DEBUG - Jaden Ivey:
  Odds: -118 (over)
  Line: 14.5
  Prediction: 17.91
  Std Dev: 5.015531433014408
  Prob Over: 0.748
  Decimal Odds: 1.85
  Breakeven: 0.541
  Simulated Mean: 17.86
  Model Prediction: 17.91

DEBUG - Jaden Ivey:
  Odds: -112 (under)
  Line: 14.5
  Prediction: 17.91
  Std Dev: 5.015531433014408
  Prob Over: 0.750
  Decimal Odds: 1.89
  Breakeven: 0.528
  Simulated Mean: 17.89
  Model Prediction: 17.91

DEBUG - Franz Wagner:
  Odds: -110 (under)
  Line: 33.5
  Prediction: 20.42
  Std Dev: 8.235424835563872
  Prob Over: 0.053
  Decimal Odds: 1.91
  Breakeven: 0.524
  Simulated Mean: 20.56
  Model Prediction: 20.42

DEBUG - Brandon Miller:
  Odds: -120 (under)
  Line: 7.5
  Prediction: 16.24
  Std Dev: 8.300889223737533
  Prob Over: 0.878
  Decimal Odds: 1.83
  Breakeven: 0.545
  Simulated Mean: 16.76
  Model Prediction: 16.24

DEBUG - LaMelo Ball:
  Odds: -140 (under)
  Line: 31.5
  Prediction: 25.16
  Std Dev: 6.719788356455548
  Prob Over: 

,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
0,John Collins,fanduel,points,27.5,102,under,16.74,0.043,0.957,0.495,93.35,0.92,0.46,0.23,"(4.8, 29.0)"
1,Clint Capela,fanduel,points,19.5,100,under,12.02,0.078,0.922,0.500,84.36,0.84,0.42,0.21,"(2.5, 22.1)"
2,Franz Wagner,fanduel,points,33.5,-110,under,20.42,0.053,0.947,0.524,80.77,0.89,0.44,0.22,"(5.2, 36.4)"
3,Brandon Miller,fanduel,points,7.5,-110,over,16.24,0.881,0.119,0.524,68.29,0.75,0.38,0.19,"(2.8, 32.7)"
4,Jayson Tatum,espnbet,points,22.5,-110,over,29.00,0.834,0.166,0.524,59.24,0.65,0.33,0.16,"(15.5, 42.0)"
5,Grayson Allen,fanduel,points,17.5,-146,under,8.66,0.089,0.911,0.593,53.53,0.78,0.39,0.20,"(0.8, 21.3)"
6,Jrue Holiday,fanduel,points,10.5,102,over,13.88,0.741,0.259,0.495,49.68,0.49,0.24,0.12,"(3.9, 24.2)"
7,LaMelo Ball,fanduel,points,31.5,-140,under,25.16,0.174,0.826,0.583,41.53,0.58,0.29,0.15,"(12.1, 38.1)"
8,Jaden Ivey,fanduel,points,14.5,-118,over,17.91,0.748,0.252,0.541,38.21,0.45,0.23,0.11,"(7.9, 27.9)"
9,Devin Booker,fanduel,points,27.5,-122,under,22.04,0.265,0.735,0.550,33.82,0.41,0.21,0.10,"(5.7, 39.0)"


## Best EVs for 2-leg parlays on prizepicks, underdogs or fanduel

In [9]:
results = prizepickspairsEV(filterData, odds, model, games, features, espnDate, stake=100, simulations=10000).sort_values(by='EV%', ascending=False).reset_index(drop=True)
print("\nTop 10 highest EV bets:")
results.head()

Processing PrizePicks pairs...

Top 10 highest EV bets:


,PLAYER 1,CATEGORY 1,LINE 1,SIDE 1,PREDICTION 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,LINE 2,SIDE 2,PREDICTION 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,TYPE,PROBABILITY,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER
0,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Clint Capela,points,19.5,under,12.02,0.080,0.920,"(2.5, 22.4)",UNDER/UNDER,0.8680,1.604,0.80,0.40,0.20
1,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Grayson Allen,points,17.5,under,8.66,0.090,0.910,"(0.9, 21.5)",UNDER/UNDER,0.8588,1.576,0.79,0.39,0.20
2,Clint Capela,points,19.5,under,12.02,0.080,0.920,"(2.5, 22.4)",Grayson Allen,points,17.5,under,8.66,0.090,0.910,"(0.9, 21.5)",UNDER/UNDER,0.8368,1.511,0.76,0.38,0.19
3,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",Jayson Tatum,points,22.5,over,29.00,0.831,0.169,"(15.6, 42.3)",UNDER/OVER,0.7840,1.352,0.68,0.34,0.17
4,Franz Wagner,points,33.5,under,20.42,0.056,0.944,"(5.4, 36.5)",LaMelo Ball,points,31.5,under,25.16,0.171,0.829,"(12.2, 38.3)",UNDER/UNDER,0.7823,1.347,0.67,0.34,0.17
